# Silver Layer - Data Cleaning & Standardization

This notebook transforms Bronze Olist e-commerce datasets into clean,
validated and analytics-ready Silver Delta tables.

## Silver Layer Responsibilities

- Enforce consistent schemas and data types
- Standardize text attributes
- Correct source column naming issues
- Handle missing values according to business meaning
- Resolve duplicate review records
- Aggregate duplicated geolocation coordinates
- Preserve valid optional null values
- Validate business rules and relationships
- Add transformation metadata
- Publish curated Delta tables for Gold-layer modeling

Transformation rules are based on findings from the Bronze data profiling notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "ecommerce_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"

print(f"Source: {BRONZE}")
print(f"Target: {SILVER}")

Source: ecommerce_lakehouse.bronze
Target: ecommerce_lakehouse.silver


In [0]:
def write_silver(df, table_name):

    target_table = f"{SILVER}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    row_count = spark.table(target_table).count()

    print(
        f"{target_table:<55} "
        f"{row_count:>12,} rows"
    )

    return row_count

In [0]:
customers = spark.table(
    f"{BRONZE}.customers"
)

silver_customers = (
    customers

    .select(
        F.col("customer_id"),
        F.col("customer_unique_id"),

        F.col("customer_zip_code_prefix")
        .cast("int")
        .alias("customer_zip_code_prefix"),

        F.lower(
            F.trim(F.col("customer_city"))
        ).alias("customer_city"),

        F.upper(
            F.trim(F.col("customer_state"))
        ).alias("customer_state"),

        F.col("_source_file")
    )

    .filter(
        F.col("customer_id").isNotNull()
    )

    .dropDuplicates(["customer_id"])

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_customers,
    "customers"
)

ecommerce_lakehouse.silver.customers                          99,441 rows


99441

In [0]:
orders = spark.table(
    f"{BRONZE}.orders"
)

silver_orders = (
    orders

    .select(
        "order_id",
        "customer_id",

        F.lower(
            F.trim(F.col("order_status"))
        ).alias("order_status"),

        F.col("order_purchase_timestamp")
        .cast("timestamp")
        .alias("order_purchase_timestamp"),

        F.col("order_approved_at")
        .cast("timestamp")
        .alias("order_approved_at"),

        F.col("order_delivered_carrier_date")
        .cast("timestamp")
        .alias("order_delivered_carrier_date"),

        F.col("order_delivered_customer_date")
        .cast("timestamp")
        .alias("order_delivered_customer_date"),

        F.col("order_estimated_delivery_date")
        .cast("timestamp")
        .alias("order_estimated_delivery_date"),

        "_source_file"
    )

    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
    )

    .dropDuplicates(["order_id"])

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_orders,
    "orders"
)

ecommerce_lakehouse.silver.orders                             99,441 rows


99441

In [0]:
order_items = spark.table(
    f"{BRONZE}.order_items"
)

silver_order_items = (
    order_items

    .select(
        "order_id",

        F.col("order_item_id")
        .cast("int")
        .alias("order_item_id"),

        "product_id",
        "seller_id",

        F.col("shipping_limit_date")
        .cast("timestamp")
        .alias("shipping_limit_date"),

        F.col("price")
        .cast("decimal(12,2)")
        .alias("price"),

        F.col("freight_value")
        .cast("decimal(12,2)")
        .alias("freight_value"),

        "_source_file"
    )

    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_item_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("seller_id").isNotNull()
        & (F.col("price") >= 0)
        & (F.col("freight_value") >= 0)
    )

    .dropDuplicates(
        ["order_id", "order_item_id"]
    )

    .withColumn(
        "item_total_value",
        F.col("price") + F.col("freight_value")
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_order_items,
    "order_items"
)

ecommerce_lakehouse.silver.order_items                       112,650 rows


112650

In [0]:
payments = spark.table(
    f"{BRONZE}.order_payments"
)

silver_payments = (
    payments

    .select(
        "order_id",

        F.col("payment_sequential")
        .cast("int")
        .alias("payment_sequential"),

        F.lower(
            F.trim(F.col("payment_type"))
        ).alias("payment_type"),

        F.col("payment_installments")
        .cast("int")
        .alias("payment_installments"),

        F.col("payment_value")
        .cast("decimal(12,2)")
        .alias("payment_value"),

        "_source_file"
    )

    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_sequential").isNotNull()
        & (F.col("payment_value") >= 0)
    )

    .dropDuplicates(
        ["order_id", "payment_sequential"]
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_payments,
    "order_payments"
)

ecommerce_lakehouse.silver.order_payments                    103,886 rows


103886

In [0]:
products = spark.table(
    f"{BRONZE}.products"
)

silver_products = (
    products

    .select(
        "product_id",

        F.coalesce(
            F.lower(
                F.trim(
                    F.col("product_category_name")
                )
            ),
            F.lit("unknown")
        ).alias("product_category_name"),

        F.col("product_name_lenght")
        .cast("int")
        .alias("product_name_length"),

        F.col("product_description_lenght")
        .cast("int")
        .alias("product_description_length"),

        F.col("product_photos_qty")
        .cast("int")
        .alias("product_photos_qty"),

        F.col("product_weight_g")
        .cast("int")
        .alias("product_weight_g"),

        F.col("product_length_cm")
        .cast("int")
        .alias("product_length_cm"),

        F.col("product_height_cm")
        .cast("int")
        .alias("product_height_cm"),

        F.col("product_width_cm")
        .cast("int")
        .alias("product_width_cm"),

        "_source_file"
    )

    .filter(
        F.col("product_id").isNotNull()
    )

    .dropDuplicates(["product_id"])

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_products,
    "products"
)

ecommerce_lakehouse.silver.products                           32,951 rows


32951

In [0]:
sellers = spark.table(
    f"{BRONZE}.sellers"
)

silver_sellers = (
    sellers

    .select(
        "seller_id",

        F.col("seller_zip_code_prefix")
        .cast("int")
        .alias("seller_zip_code_prefix"),

        F.lower(
            F.trim(F.col("seller_city"))
        ).alias("seller_city"),

        F.upper(
            F.trim(F.col("seller_state"))
        ).alias("seller_state"),

        "_source_file"
    )

    .filter(
        F.col("seller_id").isNotNull()
    )

    .dropDuplicates(["seller_id"])

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_sellers,
    "sellers"
)

ecommerce_lakehouse.silver.sellers                             3,095 rows


3095

In [0]:
categories = spark.table(
    f"{BRONZE}.product_category_translation"
)

silver_categories = (
    categories

    .select(
        F.lower(
            F.trim(
                F.col("product_category_name")
            )
        ).alias("product_category_name"),

        F.lower(
            F.trim(
                F.col("product_category_name_english")
            )
        ).alias("product_category_name_english"),

        "_source_file"
    )

    .filter(
        F.col("product_category_name").isNotNull()
    )

    .dropDuplicates(
        ["product_category_name"]
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_categories,
    "product_category_translation"
)

ecommerce_lakehouse.silver.product_category_translation           71 rows


71

In [0]:
reviews = spark.table(
    f"{BRONZE}.order_reviews"
)

review_window = (
    Window
    .partitionBy("review_id")
    .orderBy(
        F.col(
            "review_answer_timestamp"
        ).desc_nulls_last(),
        F.col(
            "review_creation_date"
        ).desc_nulls_last()
    )
)

silver_reviews = (
    reviews

    .filter(
        F.col("review_id").isNotNull()
        & F.col("order_id").isNotNull()
    )

    .withColumn(
        "_row_number",
        F.row_number().over(review_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")

    .select(
        "review_id",
        "order_id",

        F.col("review_score")
        .cast("int")
        .alias("review_score"),

        F.trim(
            F.col("review_comment_title")
        ).alias("review_comment_title"),

        F.trim(
            F.col("review_comment_message")
        ).alias("review_comment_message"),

        F.col("review_creation_date")
        .cast("timestamp")
        .alias("review_creation_date"),

        F.col("review_answer_timestamp")
        .cast("timestamp")
        .alias("review_answer_timestamp"),

        "_source_file"
    )

    .filter(
        F.col("review_score").between(1, 5)
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_reviews,
    "order_reviews"
)

ecommerce_lakehouse.silver.order_reviews                      98,410 rows


98410

In [0]:
geolocation = spark.table(
    f"{BRONZE}.geolocation"
)

silver_geolocation = (
    geolocation

    .filter(
        F.col(
            "geolocation_zip_code_prefix"
        ).isNotNull()
    )

    .withColumn(
        "geolocation_city",
        F.lower(
            F.trim(
                F.col("geolocation_city")
            )
        )
    )

    .withColumn(
        "geolocation_state",
        F.upper(
            F.trim(
                F.col("geolocation_state")
            )
        )
    )

    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    )

    .agg(
        F.avg("geolocation_lat")
        .alias("geolocation_lat"),

        F.avg("geolocation_lng")
        .alias("geolocation_lng")
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

write_silver(
    silver_geolocation,
    "geolocation"
)

ecommerce_lakehouse.silver.geolocation                        27,911 rows


27911

In [0]:
SILVER_TABLES = [
    "customers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation",
    "product_category_translation"
]

summary = []

for table_name in SILVER_TABLES:

    bronze_count = spark.table(
        f"{BRONZE}.{table_name}"
    ).count()

    silver_count = spark.table(
        f"{SILVER}.{table_name}"
    ).count()

    summary.append({
        "table": table_name,
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "rows_removed_or_aggregated":
            bronze_count - silver_count
    })

summary_df = spark.createDataFrame(summary)

display(
    summary_df.orderBy("table")
)

bronze_rows,rows_removed_or_aggregated,silver_rows,table
99441,0,99441,customers
1000163,972252,27911,geolocation
112650,0,112650,order_items
103886,0,103886,order_payments
99224,814,98410,order_reviews
99441,0,99441,orders
71,0,71,product_category_translation
32951,0,32951,products
3095,0,3095,sellers


In [0]:
SILVER_KEYS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": [
        "order_id",
        "order_item_id"
    ],
    "order_payments": [
        "order_id",
        "payment_sequential"
    ],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_translation": [
        "product_category_name"
    ]
}

validation_results = []

for table_name, keys in SILVER_KEYS.items():

    df = spark.table(
        f"{SILVER}.{table_name}"
    )

    duplicates = (
        df
        .groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    validation_results.append({
        "table": table_name,
        "duplicate_key_groups": duplicates,
        "status":
            "PASS"
            if duplicates == 0
            else "FAIL"
    })

display(
    spark.createDataFrame(
        validation_results
    )
)

duplicate_key_groups,status,table
0,PASS,customers
0,PASS,orders
0,PASS,order_items
0,PASS,order_payments
0,PASS,order_reviews
0,PASS,products
0,PASS,sellers
0,PASS,product_category_translation


In [0]:
print("=" * 70)
print("SILVER LAYER VALIDATION")
print("=" * 70)

for table_name in SILVER_TABLES:

    count = spark.table(
        f"{SILVER}.{table_name}"
    ).count()

    print(
        f"{table_name:<35} "
        f"{count:>12,}"
    )

print("=" * 70)
print("Silver layer transformation completed.")

SILVER LAYER VALIDATION
customers                                 99,441
orders                                    99,441
order_items                              112,650
order_payments                           103,886
order_reviews                             98,410
products                                  32,951
sellers                                    3,095
geolocation                               27,911
product_category_translation                  71
Silver layer transformation completed.
